<a href="https://colab.research.google.com/github/1816x/Algoritmos-Aprendizaje-Automatico/blob/main/Practica%20Tema%2011.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practica Tema 11

**Clase:** Fundamentos Algoritmos de Aprendizaje Automatico  
**Tema:** Redes Neuronales y Perceptron Multicapa (MLP)

Este notebook resuelve los cuatro retos de la practica utilizando el dataset Breast Cancer de scikit-learn.

## Reto 1. Carga, preparacion de datos y estructura base de la MLP

En este reto se carga el dataset, se divide de forma estratificada, se escalan las caracteristicas y se construye una red neuronal secuencial en Keras.

In [1]:
from tensorflow.keras import Sequential as sequential, Input as input_layer
from tensorflow.keras.layers import Dense as dense, Dropout as dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping as early_stopping_callback
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler as standard_scaler
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np
import pandas as pd

data = load_breast_cancer()

x_train, x_test, y_train, y_test = train_test_split(
    data.data,
    data.target,
    test_size=0.2,
    stratify=data.target,
    random_state=42
)

scaler = standard_scaler()
x_train_s = scaler.fit_transform(x_train)
x_test_s = scaler.transform(x_test)

base_model = sequential([
    input_layer(shape=(x_train_s.shape[1],)),
    dense(32, activation="relu"),
    dense(16, activation="relu"),
    dense(1, activation="sigmoid")
])

base_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("base model compiled successfully")
print("total features:", x_train_s.shape[1])
base_model.summary()

base model compiled successfully
total features: 30


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,537 (6.00 KB)

 Trainable params: 1,537 (6.00 KB)

 Non-trainable params: 0 (0.00 B)

## Reto 2. Interpretacion matricial y propagacion hacia adelante

Se extraen los pesos y sesgos de la primera capa oculta. Despues se calcula manualmente la combinacion lineal $z = xw + b$ y se aplica la funcion ReLU.

In [2]:
first_hidden_layer = base_model.layers[0]
weights, bias = first_hidden_layer.get_weights()

x_sample = x_train_s[:5]
z = np.dot(x_sample, weights) + bias
relu_output = np.maximum(0, z)

print("weights shape:", weights.shape)
print("bias shape:", bias.shape)
print("sample shape:", x_sample.shape)
print("z shape:", z.shape)
print("relu output shape:", relu_output.shape)

print("\nfirst row of z:")
print(z[0])

print("\nfirst row after relu:")
print(relu_output[0])

weights shape: (30, 32)
bias shape: (32,)
sample shape: (5, 30)
z shape: (5, 32)
relu output shape: (5, 32)

first row of z:
[-0.05201572 -0.86284631  0.00961413  0.37960753  0.46379372 -1.66294964
  0.15620009  0.41852191 -0.2746826   0.02537314  0.53583118 -0.58460845
 -0.81225397 -1.74881148  0.13468552  0.01515966 -0.96078278 -1.11016027
 -0.60922866  0.53222472 -0.76549202 -0.71975037 -0.86980574  0.52454576
 -0.69013813 -0.26725466  1.3535675  -0.61129548  0.48313241 -0.50461657
  0.66501744  0.79903686]

first row after relu:
[0.         0.         0.00961413 0.37960753 0.46379372 0.
 0.15620009 0.41852191 0.         0.02537314 0.53583118 0.
 0.         0.         0.13468552 0.01515966 0.         0.
 0.         0.53222472 0.         0.         0.         0.52454576
 0.         0.         1.3535675  0.         0.48313241 0.
 0.66501744 0.79903686]


La operacion matricial combina cada observacion con los pesos aprendibles de la capa y agrega el sesgo. ReLU transforma los valores negativos en cero y mantiene los positivos.

## Reto 3. Modificacion arquitectonica para mitigar sobreajuste

Se aumenta la capacidad de la red y se agregan regularizacion L2 y capas Dropout para reducir el riesgo de sobreajuste.

In [3]:
regularized_model = sequential([
    input_layer(shape=(x_train_s.shape[1],)),
    dense(64, activation="relu", kernel_regularizer=l2(0.001)),
    dropout(0.30),
    dense(32, activation="relu", kernel_regularizer=l2(0.001)),
    dropout(0.20),
    dense(16, activation="relu", kernel_regularizer=l2(0.001)),
    dense(1, activation="sigmoid")
])

regularized_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("regularized model compiled successfully")
regularized_model.summary()

regularized model compiled successfully


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │         1,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,609 (18.00 KB)

 Trainable params: 4,609 (18.00 KB)

 Non-trainable params: 0 (0.00 B)

## Reto 4. Entrenamiento con parada temprana y analisis operativo

Se usa EarlyStopping para detener el entrenamiento cuando la perdida de validacion deja de mejorar. Posteriormente se prueban distintos umbrales de decision y se selecciona el que maximiza F1.

In [4]:
early_stopping = early_stopping_callback(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = regularized_model.fit(
    x_train_s,
    y_train,
    validation_split=0.2,
    epochs=150,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/150
12/12 ━━━━━━━━━━━━━━━━━━━━ 4s 45ms/step - accuracy: 0.6978 - loss: 0.7567 - val_accuracy: 0.9341 - val_loss: 0.5421
Epoch 2/150
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.8736 - loss: 0.5215 - val_accuracy: 0.9341 - val_loss: 0.4151
Epoch 3/150
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9368 - loss: 0.3899 - val_accuracy: 0.9341 - val_loss: 0.3377
Epoch 4/150
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9451 - loss: 0.3228 - val_accuracy: 0.9451 - val_loss: 0.2909
Epoch 5/150
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9588 - loss: 0.2750 - val_accuracy: 0.9451 - val_loss: 0.2592
Epoch 6/150
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9423 - loss: 0.2597 - val_accuracy: 0.9560 - val_loss: 0.2358
Epoch 7/150
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9588 - loss: 0.2229 - val_accuracy: 0.9451 - val_loss: 0.2173
Epoch 8/150
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9560 - loss: 0.2336 - val_accuracy: 0.

In [5]:
test_probabilities = regularized_model.predict(x_test_s).ravel()

thresholds = np.arange(0.30, 0.71, 0.05)
results = []

for threshold in thresholds:
    predictions = (test_probabilities >= threshold).astype(int)

    results.append({
        "threshold": round(float(threshold), 2),
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions, zero_division=0),
        "recall": recall_score(y_test, predictions, zero_division=0),
        "f1": f1_score(y_test, predictions, zero_division=0)
    })

results_df = pd.DataFrame(results)
results_df

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step


,threshold,accuracy,precision,recall,f1
0,0.30,0.95614,0.985507,0.944444,0.964539
1,0.35,0.95614,0.985507,0.944444,0.964539
2,0.40,0.95614,0.985507,0.944444,0.964539
3,0.45,0.95614,0.985507,0.944444,0.964539
4,0.50,0.95614,0.985507,0.944444,0.964539
5,0.55,0.95614,0.985507,0.944444,0.964539
6,0.60,0.95614,0.985507,0.944444,0.964539
7,0.65,0.95614,0.985507,0.944444,0.964539
8,0.70,0.95614,0.985507,0.944444,0.964539


In [6]:
best_row = results_df.loc[results_df["f1"].idxmax()]
best_threshold = best_row["threshold"]

print("best threshold based on f1:", best_threshold)
print("accuracy:", round(best_row["accuracy"], 4))
print("precision:", round(best_row["precision"], 4))
print("recall:", round(best_row["recall"], 4))
print("f1:", round(best_row["f1"], 4))

best threshold based on f1: 0.3
accuracy: 0.9561
precision: 0.9855
recall: 0.9444
f1: 0.9645


## Conclusion

La practica muestra el flujo completo de una MLP para clasificacion binaria: preparacion de datos, propagacion hacia adelante, regularizacion, entrenamiento con parada temprana y ajuste del umbral de decision. El uso de L2, Dropout y EarlyStopping ayuda a controlar el sobreajuste, mientras que la evaluacion de distintos umbrales permite adaptar el comportamiento del clasificador a la metrica objetivo.